# Scoring Engine — Validation Notebook

**Goal:** Verify the tier-aware scoring engine produces sensible picks.  
**Checks:**
1. Weights from config.SIGNAL_WEIGHTS (sum to 1.0 per tier)
2. Rankings are within-tier only
3. Quality gate excludes ~15-30% of small caps
4. Compare top 15 against v1's daily picks
5. VIX regime allocation is correct

In [ ]:
import sys, os
os.chdir(os.path.expanduser("~/alpha-signal-v2"))
sys.path.insert(0, ".")

from config import SIGNAL_WEIGHTS
import pandas as pd

# Check 1: Weights sum to 1.0
for tier, weights in SIGNAL_WEIGHTS.items():
    total = sum(weights.values())
    print(f"{tier}: {dict(weights)} → sum={total:.2f} {'✓' if abs(total - 1.0) < 0.01 else '✗'}")

In [ ]:
# Check 2: Run screener and inspect picks
from scoring.screener import _load_signals, score_universe, select_picks

df = _load_signals()
df = score_universe(df)

# Within-tier ranking check
for tier in ["LARGE", "MID", "SMALL"]:
    t = df[df["cap_tier"] == tier]
    scored = t["final_score"].notna().sum()
    rank_max = t["rank"].max()
    print(f"{tier}: {scored} scored, max_rank={rank_max}")
    assert rank_max <= scored + 1, f"Rank exceeds scored count in {tier}!"

print("\nRanking is within-tier ✓")

# Top 5 per tier
picks = select_picks(df, {"LARGE": 5, "MID": 5, "SMALL": 5})
print(f"\nTop 5 per tier:")
print(picks[["rank", "cap_tier", "sid", "ticker", "sector", "final_score"]].to_string(index=False))

In [ ]:
# Check 3: Quality gate stats
from scoring.quality_gate import compute_quality_gate

qg = compute_quality_gate()
print(f"Quality Gate (SMALL only):")
print(qg["gate_status"].value_counts())
print(f"\nExclusion rate: {(qg['gate_status']=='EXCLUDED').mean()*100:.1f}%")

In [ ]:
# Check 4: Compare with v1 picks
v1_picks = pd.read_csv(os.path.expanduser("~/alpha-signal/output/daily_picks.csv"))
if len(v1_picks) > 0:
    v1_top15 = set(v1_picks.nsmallest(15, "rank")["sid"]) if "rank" in v1_picks.columns else set(v1_picks.head(15)["sid"])
    v2_picks_15 = select_picks(df, {"LARGE": 5, "MID": 5, "SMALL": 5})
    v2_top15 = set(v2_picks_15["sid"])
    
    overlap = v1_top15 & v2_top15
    print(f"v1 top 15: {v1_top15}")
    print(f"v2 top 15: {v2_top15}")
    print(f"Overlap: {len(overlap)}/{min(len(v1_top15), 15)} = {len(overlap)/min(len(v1_top15),15)*100:.0f}%")
else:
    print("No v1 picks file found — skip comparison")

In [ ]:
# Check 5: VIX regime
from scoring.regime import compute
compute(dry_run=True)

## Save picks to DB (only if validation passed)

In [ ]:
# from scoring.screener import compute
# compute(dry_run=False)